# 09 · TTF Price Analysis

Volatility, distribution, seasonality and regime detection for TTF flat price.

**Data sources:**
- TTF forward curve: `data/raw/ttf_curve.csv`
- EU storage: `data/processed/eu_aggregate_full.parquet`

**Configuration:** set `ANALYSIS_DATE` and `START_DATE` in cell 0.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import date
import calendar
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f"✅ Root: {_c}"); break

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════
ANALYSIS_DATE = None          # None = last data point | or date(2024,9,1)
START_DATE    = "2020-01-01"
# ═══════════════════════════════════════════════════════════════════

from src.analysis.price_analysis import (
    rolling_volatility, garch_volatility, price_distribution_by_fill,
    seasonal_price_pattern, price_regime_detection
)

# ── Load data ─────────────────────────────────────────────────────
ttf_path = Path("data/raw/ttf_curve.csv")
storage_path = Path("data/processed/eu_aggregate_full.parquet")

ttf_df = pd.read_csv(ttf_path, index_col=0, parse_dates=True) if ttf_path.exists() else pd.DataFrame()
storage_df = pd.read_parquet(storage_path) if storage_path.exists() else pd.DataFrame()

if not ttf_df.empty:
    ttf_df = ttf_df[ttf_df.index >= START_DATE].sort_index()
if not storage_df.empty:
    storage_df = storage_df[storage_df.index >= START_DATE].sort_index()


# Strip timezone info — TTF CSV is tz-aware (UTC), parquet is tz-naive
for _df in [ttf_df, storage_df]:
    if not _df.empty:
        _df.index = pd.to_datetime(_df.index).tz_localize(None)
analysis_date = (
    ANALYSIS_DATE
    if ANALYSIS_DATE
    else (ttf_df.index[-1].date() if not ttf_df.empty else date.today())
)
print(f"Analysis date: {analysis_date}")
print(f"TTF rows: {len(ttf_df)}, Storage rows: {len(storage_df)}")

## 1 · Rolling Volatility with Fill Rate Overlay

In [ ]:
if ttf_df.empty:
    print("⚠️  No TTF data. Run notebook 01 / 07 first.")
else:
    vol_df = rolling_volatility(ttf_df["M1"], windows=[5, 21, 63])

    # M1 delivery label per trade date (M1 = next calendar month)
    _m1_del = np.array([
        f"{calendar.month_abbr[(dt.month % 12) + 1]}'{str(dt.year + dt.month // 12)[-2:]}"
        for dt in vol_df.index
    ])

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["TTF M1 Rolling Volatility (annualised %)", "EU Storage Fill Rate (%)"],
                        vertical_spacing=0.08, row_heights=[0.65, 0.35])

    colors = {"vol_5d": "#ef4444", "vol_21d": "#f97316", "vol_63d": "#3b82f6"}
    labels = {"vol_5d": "5d vol", "vol_21d": "21d vol", "vol_63d": "63d vol"}

    for col, color in colors.items():
        _lbl = labels[col]
        fig.add_trace(go.Scatter(
            x=vol_df.index, y=vol_df[col],
            name=_lbl, line=dict(color=color, width=1.5), mode="lines",
            customdata=_m1_del,
            hovertemplate=(
                f'%{{x|%d %b %Y}}<br>M1 (%{{customdata}}) {_lbl}: %{{y:.1f}}%<extra></extra>'
            )), row=1, col=1)

    if not storage_df.empty and "full" in storage_df.columns:
        fig.add_trace(go.Scatter(x=storage_df.index, y=storage_df["full"],
                                 name="Fill %", fill="tozeroy",
                                 line=dict(color="#22c55e", width=1),
                                 fillcolor="rgba(34,197,94,0.15)"), row=2, col=1)

    fig.update_layout(height=550, template="plotly_white",
                      title="TTF M1 Volatility & EU Gas Fill Rate",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()
    print(vol_df.tail(5).round(1))

## 2 · GARCH(1,1) Conditional Volatility

In [ ]:
if ttf_df.empty:
    print("⚠️  No TTF data.")
else:
    try:
        garch_df = garch_volatility(ttf_df["M1"])

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            subplot_titles=["TTF M1 Price (€/MWh)", "GARCH(1,1) Conditional Volatility (% annualised)"],
                            vertical_spacing=0.08)

        fig.add_trace(go.Scatter(x=ttf_df.index, y=ttf_df["M1"],
                                 name="M1 Price", line=dict(color="#6366f1", width=1.5)), row=1, col=1)

        fig.add_trace(go.Scatter(x=garch_df.index, y=garch_df["cond_vol"],
                                 name="GARCH vol", fill="tozeroy",
                                 line=dict(color="#f59e0b", width=1.5),
                                 fillcolor="rgba(245,158,11,0.15)"), row=2, col=1)

        fig.update_layout(height=500, template="plotly_white",
                          title="GARCH(1,1) Conditional Volatility — TTF M1",
                          legend=dict(orientation="h", y=-0.12))
        fig.show()
        print(f"Latest GARCH vol: {garch_df['cond_vol'].iloc[-1]:.1f}%")
    except ImportError as e:
        print(f"⚠️  {e}  →  pip install arch")

## 3 · Price Distribution by Fill Bucket (Violin)

In [ ]:
if ttf_df.empty or storage_df.empty:
    print("⚠️  Need both TTF and storage data.")
else:
    import plotly.express as px
    stats = price_distribution_by_fill(ttf_df, storage_df, n_buckets=5)

    # Rebuild raw data with bucket labels for violin
    merged = ttf_df[["M1"]].join(storage_df[["full"]], how="inner").dropna()
    merged["bucket"] = pd.cut(merged["full"], bins=5)
    merged["bucket_label"] = merged["bucket"].astype(str)

    fig = px.violin(merged, x="bucket_label", y="M1",
                    box=True, points="outliers",
                    title="TTF M1 Price Distribution by EU Fill Rate Bucket",
                    labels={"bucket_label": "Fill Rate Bucket", "M1": "TTF M1 (€/MWh)"},
                    color="bucket_label",
                    color_discrete_sequence=px.colors.sequential.Viridis_r)
    fig.update_layout(template="plotly_white", height=480, showlegend=False)
    fig.show()
    print(stats[["count", "mean", "median", "p10", "p90"]].round(1))

## 4 · Seasonal Price Patterns

In [ ]:
if ttf_df.empty:
    print("⚠️  No TTF data.")
else:
    patterns = seasonal_price_pattern(ttf_df, col="M1")
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    # Monthly average
    by_month = patterns["by_month"]
    by_month.index = [month_names[i-1] for i in by_month.index]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=by_month.index, y=by_month["avg_price"],
                         marker_color="#6366f1", name="Avg M1 Price"))
    fig.update_layout(template="plotly_white", height=380,
                      title="Average TTF M1 Price by Calendar Month",
                      yaxis_title="€/MWh", xaxis_title="Month")
    fig.show()

    # Week-of-year heatmap
    by_woy = patterns["by_week_of_year"]
    fig2 = go.Figure(go.Scatter(x=by_woy.index, y=by_woy["avg_price"],
                                mode="lines+markers", line=dict(color="#f97316"),
                                name="Avg M1 by week"))
    fig2.update_layout(template="plotly_white", height=350,
                       title="Average TTF M1 Price by Week of Year",
                       xaxis_title="ISO Week", yaxis_title="€/MWh")
    fig2.show()

## 5 · Price Regime Detection (HMM)

In [ ]:
if ttf_df.empty:
    print("⚠️  No TTF data.")
else:
    try:
        regime_df = price_regime_detection(ttf_df, col="M1", n_regimes=3)
        palette = {"Low": "#22c55e", "Medium": "#f59e0b", "High": "#ef4444",
                   "Very High": "#7c3aed", "Extreme": "#1e1b4b"}

        fig = go.Figure()
        for label, grp in regime_df.groupby("regime_label"):
            fig.add_trace(go.Scatter(
                x=grp.index, y=grp["price"], mode="markers",
                marker=dict(color=palette.get(label, "#94a3b8"), size=3, opacity=0.7),
                name=label
            ))
        # Price line overlay
        fig.add_trace(go.Scatter(x=regime_df.index, y=regime_df["price"],
                                 mode="lines", line=dict(color="rgba(0,0,0,0.2)", width=0.8),
                                 showlegend=False))
        fig.update_layout(template="plotly_white", height=420,
                          title="TTF M1 Price Regime Detection (Gaussian HMM, 3 states)",
                          yaxis_title="€/MWh",
                          legend=dict(orientation="h", y=-0.12))
        fig.show()
        print(regime_df["regime_label"].value_counts())
    except ImportError as e:
        print(f"⚠️  {e}  →  pip install hmmlearn")